# Airbnb Market Intelligence

# 08 - Feature Engineering

## Objectives

- Load cleaned dataset
- Create new analytical features
- Prepare data for Machine Learning
- Save processed dataset

In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/cleaned/listings_cleaned.csv")

print(df.shape)

df.head()

(279712, 32)


,listing_id,name,host_id,host_since,host_location,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_total_listings_count,...,minimum_nights,maximum_nights,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,instant_bookable
0,281420,"Beautiful Flat in le Village Montmartre, Paris",1466919,2011-12-03,"Paris, Ile-de-France, France",Unknown,1.0,0.98,f,1.0,...,2,1125,100.0,10.0,10.0,10.0,10.0,10.0,10.0,f
1,3705183,39 mÃÂ² Paris (Sacre CÃ âur),10328771,2013-11-29,"Paris, Ile-de-France, France",Unknown,1.0,0.98,f,1.0,...,2,1125,100.0,10.0,10.0,10.0,10.0,10.0,10.0,f
2,4082273,"Lovely apartment with Terrace, 60m2",19252768,2014-07-31,"Paris, Ile-de-France, France",Unknown,1.0,0.98,f,1.0,...,2,1125,100.0,10.0,10.0,10.0,10.0,10.0,10.0,f
3,4797344,Cosy studio (close to Eiffel tower),10668311,2013-12-17,"Paris, Ile-de-France, France",Unknown,1.0,0.98,f,1.0,...,2,1125,100.0,10.0,10.0,10.0,10.0,10.0,10.0,f
4,4823489,Close to Eiffel Tower - Beautiful flat : 2 rooms,24837558,2014-12-14,"Paris, Ile-de-France, France",Unknown,1.0,0.98,f,1.0,...,2,1125,100.0,10.0,10.0,10.0,10.0,10.0,10.0,f


In [3]:
df["host_since"] = pd.to_datetime(df["host_since"])

In [4]:
today = pd.Timestamp.today()

df["host_experience_years"] = (
    today - df["host_since"]
).dt.days / 365

In [5]:
df["amenities_count"] = (
    df["amenities"]
    .astype(str)
    .str.count(",") + 1
)

In [6]:
df["price_category"] = pd.cut(

    df["price"],

    bins=[0,100,300,700,100000],

    labels=[
        "Budget",
        "Standard",
        "Premium",
        "Luxury"
    ]
)

In [7]:
df["rating_category"] = pd.cut(

    df["review_scores_rating"],

    bins=[0,6,8,9,10],

    labels=[
        "Low",
        "Average",
        "Good",
        "Excellent"
    ]
)

In [8]:
df["bedroom_density"] = (

    df["accommodates"]

    /

    df["bedrooms"]

)

In [9]:
df["price_per_guest"] = (
    df["price"]
    /
    df["accommodates"]
)

In [10]:
score_cols = [
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",
    "review_scores_location",
    "review_scores_value"
]

df["overall_review_score"] = df[score_cols].mean(axis=1)

In [13]:
# Convert t/f to True/False
bool_cols = [
    "host_identity_verified",
    "host_has_profile_pic",
    "host_is_superhost"
]

for col in bool_cols:
    df[col] = df[col].map({"t": True, "f": False})

# Create Host Quality Score
df["host_quality"] = (
    df["host_identity_verified"].astype(int)
    + df["host_has_profile_pic"].astype(int)
    + df["host_is_superhost"].astype(int)
)

# Check the result
df[[
    "host_identity_verified",
    "host_has_profile_pic",
    "host_is_superhost",
    "host_quality"
]].head()

,host_identity_verified,host_has_profile_pic,host_is_superhost,host_quality
0,False,True,False,1
1,True,True,False,2
2,False,True,False,1
3,True,True,False,2
4,False,True,False,1


In [14]:
review_count = (

    pd.read_csv("../data/cleaned/reviews_cleaned.csv")

    .groupby("listing_id")

    .size()

)

df["review_count"] = (

    df["listing_id"]

    .map(review_count)

)

df["review_count"] = df["review_count"].fillna(0)

In [15]:
df["estimated_monthly_revenue"] = (

    df["price"]

    *

    df["review_count"]

)

In [16]:
df["host_size"] = pd.cut(

    df["host_total_listings_count"],

    bins=[0,1,5,20,10000],

    labels=[
        "Single",
        "Small",
        "Medium",
        "Large"
    ]
)

In [17]:
new_cols = [

    "host_experience_years",

    "amenities_count",

    "price_category",

    "rating_category",

    "bedroom_density",

    "price_per_guest",

    "overall_review_score",

    "host_quality",

    "review_count",

    "estimated_monthly_revenue",

    "host_size"

]

df[new_cols].head()

,host_experience_years,amenities_count,price_category,rating_category,bedroom_density,price_per_guest,overall_review_score,host_quality,review_count,estimated_monthly_revenue,host_size
0,14.572603,5,Budget,NaN,2.0,26.5,10.0,1,2.0,106.0,Single
1,12.580822,8,Standard,NaN,2.0,60.0,10.0,2,6.0,720.0,Single
2,11.912329,6,Budget,NaN,2.0,44.5,10.0,1,1.0,89.0,Single
3,12.531507,5,Budget,NaN,2.0,29.0,10.0,2,1.0,58.0,Single
4,11.539726,12,Budget,NaN,2.0,30.0,10.0,1,1.0,60.0,Single


In [18]:
df.to_csv(

    "../data/processed/airbnb_feature_engineered.csv",

    index=False

)

print("Dataset Saved Successfully")

Dataset Saved Successfully
